In [2]:
# ═══════════════════════════════════════════════════
# CELLULE 1 — Imports + Chargement du modèle
# ═══════════════════════════════════════════════════
import pandas as pd
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



In [3]:
# Charger le modèle XGBoost
model    = joblib.load('../models/xgboost_lap_predictor.pkl')
le_driver = joblib.load('../models/le_driver.pkl')
le_race   = joblib.load('../models/le_race.pkl')

with open('../models/model_metadata.json') as f:
    metadata = json.load(f)

features     = metadata['features']
compound_map = metadata['compound_map']

print("✅ Modèle chargé")
print(f"   Type    : {metadata['model_type']}")
print(f"   MAE     : {metadata['performance']['MAE']}s")
print(f"   R²      : {metadata['performance']['R2']}")
print(f"\n✅ Pilotes disponibles:")
print(f"   {metadata['driver_classes']}")
print(f"\n✅ Circuits disponibles:")
for r in metadata['race_classes']:
    print(f"   - {r}")

✅ Modèle chargé
   Type    : XGBoostRegressor
   MAE     : 0.4138s
   R²      : 0.9958

✅ Pilotes disponibles:
   ['ALB', 'ALO', 'ANT', 'BEA', 'BOR', 'BOT', 'COL', 'DOO', 'GAS', 'HAD', 'HAM', 'HUL', 'LAW', 'LEC', 'MAG', 'NOR', 'OCO', 'PER', 'PIA', 'RIC', 'RUS', 'SAI', 'SAR', 'STR', 'TSU', 'VER', 'ZHO']

✅ Circuits disponibles:
   - Abu Dhabi Grand Prix
   - Australian Grand Prix
   - Austrian Grand Prix
   - Azerbaijan Grand Prix
   - Bahrain Grand Prix
   - Belgian Grand Prix
   - British Grand Prix
   - Canadian Grand Prix
   - Chinese Grand Prix
   - Dutch Grand Prix
   - Emilia Romagna Grand Prix
   - Hungarian Grand Prix
   - Italian Grand Prix
   - Japanese Grand Prix
   - Las Vegas Grand Prix
   - Mexico City Grand Prix
   - Miami Grand Prix
   - Monaco Grand Prix
   - Qatar Grand Prix
   - Saudi Arabian Grand Prix
   - Singapore Grand Prix
   - Spanish Grand Prix
   - São Paulo Grand Prix
   - United States Grand Prix


In [4]:
# ═══════════════════════════════════════════════════
# CELLULE 2 — Feature Engineering (VERSION FINALE)
# ═══════════════════════════════════════════════════
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/all_laps_2024_2025.csv')

print(f"✅ CSV chargé: {df.shape}")
print(f"   Colonnes: {df.columns.tolist()}")
print(f"   Courses : {df['Race'].nunique()}")
print(f"   Années  : {sorted(df['Year'].unique())}")

# ─────────────────────────────────────────
# 1. Encodages
# ─────────────────────────────────────────
compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}
df['CompoundEncoded'] = df['Compound'].map(compound_map)

le_driver = LabelEncoder()
df['DriverEncoded'] = le_driver.fit_transform(df['Driver'])

le_race = LabelEncoder()
df['RaceEncoded'] = le_race.fit_transform(df['Race'])

print(f"\n✅ Encodages:")
print(f"   Compounds : {compound_map}")
print(f"   Pilotes   : {len(le_driver.classes_)} ({list(le_driver.classes_)})")
print(f"   Circuits  : {len(le_race.classes_)}")

# ─────────────────────────────────────────
# 2. Circuit Baseline (CRITIQUE)
#    Normalise les différences inter-circuits
#    Monaco 78s vs Monza 85s vs Bahrain 96s
# ─────────────────────────────────────────
df['CircuitBaseline'] = df.groupby(['Race', 'Year'])['LapTime'].transform('median')
df['LapTimeDelta']    = df['LapTime'] - df['CircuitBaseline']

print(f"\n✅ Circuit Baseline calculé:")
print(f"   Std LapTime original : {df['LapTime'].std():.2f}s")
print(f"   Std LapTimeDelta     : {df['LapTimeDelta'].std():.2f}s  ← doit être ~3-4s")
print(f"   Median LapTimeDelta  : {df['LapTimeDelta'].median():.3f}s  ← doit être ~0")

print(f"\n   Baseline par circuit (sample):")
sample = df.groupby('Race')['CircuitBaseline'].first().sort_values()
for race, baseline in sample.items():
    print(f"      {race:35s}: {baseline:.2f}s")

# ─────────────────────────────────────────
# 3. Features dérivées
# ─────────────────────────────────────────

# Dégradation non-linéaire
df['TyreLifeSquared'] = df['TyreLife'] ** 2

# Interaction composé × âge pneu
df['CompoundTireInteraction'] = df['CompoundEncoded'] * df['TyreLife']

# Pneu neuf (tours 1-5 = comportement différent)
df['IsNewTire'] = (df['TyreLife'] <= 5).astype(int)

# Fuel effect
df['FuelEffect'] = (55 - df['LapNumber']).clip(lower=0) * 0.035

# Stint progress (0 → 1)
max_tyre = df.groupby(
    ['Driver', 'Race', 'Year', 'Stint']
)['TyreLife'].transform('max')
df['StintProgress'] = df['TyreLife'] / max_tyre.clip(lower=1)

# ─────────────────────────────────────────
# 4. Vérification finale
# ─────────────────────────────────────────
all_features = [
    'CircuitBaseline', 'LapTimeDelta',
    'TyreLifeSquared', 'CompoundTireInteraction',
    'IsNewTire', 'FuelEffect', 'StintProgress',
    'CompoundEncoded', 'DriverEncoded', 'RaceEncoded'
]

print(f"\n✅ Features créées:")
for col in all_features:
    nan_count = df[col].isna().sum()
    status = "✅" if nan_count == 0 else "❌"
    print(f"   {status} {col:30s}: {nan_count} NaN")

print(f"\n📊 Shape finale: {df.shape}")
print(f"   NaN total: {df.isnull().sum().sum()}")

✅ CSV chargé: (45172, 10)
   Colonnes: ['Driver', 'LapTime', 'Compound', 'TyreLife', 'LapNumber', 'Stint', 'Position', 'TrackStatus', 'Race', 'Year']
   Courses : 24
   Années  : [np.int64(2024), np.int64(2025)]

✅ Encodages:
   Compounds : {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}
   Pilotes   : 27 (['ALB', 'ALO', 'ANT', 'BEA', 'BOR', 'BOT', 'COL', 'DOO', 'GAS', 'HAD', 'HAM', 'HUL', 'LAW', 'LEC', 'MAG', 'NOR', 'OCO', 'PER', 'PIA', 'RIC', 'RUS', 'SAI', 'SAR', 'STR', 'TSU', 'VER', 'ZHO'])
   Circuits  : 24

✅ Circuit Baseline calculé:
   Std LapTime original : 10.74s
   Std LapTimeDelta     : 3.49s  ← doit être ~3-4s
   Median LapTimeDelta  : 0.000s  ← doit être ~0

   Baseline par circuit (sample):
      Austrian Grand Prix                : 71.20s
      São Paulo Grand Prix               : 74.83s
      Dutch Grand Prix                   : 76.36s
      Canadian Grand Prix                : 78.79s
      Monaco Grand Prix                  : 79.50s
      Spanish Grand Prix                 : 80.91

In [5]:
# ═══════════════════════════════════════════════════
# CELLULE 3 — Strategy Simulator
# ═══════════════════════════════════════════════════

class RaceStrategySimulator:
    """Simulateur de stratégie F1 — propulsé par XGBoost R²=0.9958"""

    def __init__(self, race_name, year=2025,
                 total_laps=55, pit_stop_time=22.0):
        self.race_name     = race_name
        self.year          = year
        self.total_laps    = total_laps
        self.pit_stop_time = pit_stop_time
        self.baseline      = get_circuit_baseline(race_name, year)
        print(f"✅ Simulateur: {race_name} {year}")
        print(f"   Baseline : {self.baseline:.2f}s")
        print(f"   Tours    : {total_laps}")
        print(f"   Pit stop : {pit_stop_time}s")

    def simulate_strategy(self, driver, strategy,
                          position=5, verbose=False):
        """
        Simule une stratégie complète

        strategy = [
            {'compound': 'MEDIUM', 'laps': 28},
            {'compound': 'HARD',   'laps': 27}
        ]
        """
        total_time  = 0
        current_lap = 1
        stints_info = []

        for i, stint in enumerate(strategy):
            stint_times = []

            for tire_age in range(1, stint['laps'] + 1):
                lt = predict_lap_time(
                    driver     = driver,
                    compound   = stint['compound'],
                    tire_age   = tire_age,
                    lap_number = current_lap,
                    race_name  = self.race_name,
                    year       = self.year,
                    position   = position,
                    total_laps = self.total_laps
                )
                stint_times.append(lt)
                current_lap += 1

            stint_total = sum(stint_times)
            total_time += stint_total

            stints_info.append({
                'stint'    : i + 1,
                'compound' : stint['compound'],
                'laps'     : stint['laps'],
                'time'     : stint_total,
                'avg_lap'  : stint_total / stint['laps'],
                'lap_times': stint_times
            })

            if verbose:
                print(f"   Stint {i+1} ({stint['compound']:6s}): "
                      f"{stint['laps']} laps | "
                      f"avg {stint_total/stint['laps']:.3f}s | "
                      f"total {stint_total:.1f}s")

            # Pit stop (sauf dernier stint)
            if i < len(strategy) - 1:
                total_time += self.pit_stop_time
                if verbose:
                    print(f"   🔧 Pit stop: +{self.pit_stop_time}s")

        return {
            'total_time'    : total_time,
            'total_min'     : total_time / 60,
            'avg_lap'       : total_time / self.total_laps,
            'pit_stops'     : len(strategy) - 1,
            'stints'        : stints_info,
            'compounds'     : ' → '.join(s['compound'] for s in strategy)
        }

    def compare_strategies(self, driver, strategies_dict, position=5):
        """Compare plusieurs stratégies, retourne un DataFrame trié"""
        rows = []
        for name, strat in strategies_dict.items():
            res = self.simulate_strategy(driver, strat,
                                         position, verbose=False)
            rows.append({
                'Strategy'       : name,
                'Compounds'      : res['compounds'],
                'Pit Stops'      : res['pit_stops'],
                'Total Time (s)' : round(res['total_time'], 2),
                'Total Time (min)': round(res['total_min'],  2),
                'Avg Lap (s)'    : round(res['avg_lap'],     3)
            })

        df_res = pd.DataFrame(rows).sort_values('Total Time (s)')
        df_res['Delta (s)'] = (
            df_res['Total Time (s)'] - df_res['Total Time (s)'].iloc[0]
        ).round(2)
        return df_res


print("✅ Classe RaceStrategySimulator définie")

✅ Classe RaceStrategySimulator définie


In [6]:
# ═══════════════════════════════════════════════════
# CELLULE 4 — Test : Bahrain GP 2025
# ═══════════════════════════════════════════════════

sim = RaceStrategySimulator(
    race_name  = 'Bahrain Grand Prix',
    year       = 2025,
    total_laps = 57,
    pit_stop_time = 22.0
)

strategies = {
    '1-stop MEDIUM→HARD (early, lap 20)': [
        {'compound': 'MEDIUM', 'laps': 20},
        {'compound': 'HARD',   'laps': 37}
    ],
    '1-stop MEDIUM→HARD (mid, lap 28)': [
        {'compound': 'MEDIUM', 'laps': 28},
        {'compound': 'HARD',   'laps': 29}
    ],
    '1-stop MEDIUM→HARD (late, lap 35)': [
        {'compound': 'MEDIUM', 'laps': 35},
        {'compound': 'HARD',   'laps': 22}
    ],
    '1-stop SOFT→HARD': [
        {'compound': 'SOFT', 'laps': 22},
        {'compound': 'HARD', 'laps': 35}
    ],
    '2-stops SOFT→MEDIUM→HARD': [
        {'compound': 'SOFT',   'laps': 18},
        {'compound': 'MEDIUM', 'laps': 20},
        {'compound': 'HARD',   'laps': 19}
    ],
    '2-stops MEDIUM→MEDIUM→HARD': [
        {'compound': 'MEDIUM', 'laps': 18},
        {'compound': 'MEDIUM', 'laps': 20},
        {'compound': 'HARD',   'laps': 19}
    ]
}

print("\n📊 Comparaison stratégies — VER — Bahrain 2025\n")
results_strat = sim.compare_strategies('VER', strategies, position=1)
print(results_strat.to_string(index=False))

best = results_strat.iloc[0]
print(f"\n🏆 Stratégie optimale : {best['Strategy']}")
print(f"   Compounds : {best['Compounds']}")
print(f"   Temps     : {best['Total Time (min)']:.2f} min")
print(f"   Delta vs worst : +{results_strat['Delta (s)'].iloc[-1]:.1f}s")

NameError: name 'get_circuit_baseline' is not defined

# ═══════════════════════════════════════════════════
# CELLULE 5 — Pit Window Optimization
# ═══════════════════════════════════════════════════

df_opt, optimal = sim.optimize_pit_window(
    driver     = 'VER',
    compound_1 = 'MEDIUM',
    compound_2 = 'HARD',
    search_range = (15, 40),
    position   = 1
)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Courbe pit window
ax1 = axes[0]
min_time = df_opt['total_time'].min()

colors_pit = []
for t in df_opt['total_time']:
    if t == min_time:
        colors_pit.append('green')
    elif t <= min_time + 1:
        colors_pit.append('yellow')
    elif t <= min_time + 3:
        colors_pit.append('orange')
    else:
        colors_pit.append('red')

ax1.plot(df_opt['pit_lap'], df_opt['total_minutes'],
         linewidth=3, color='steelblue', alpha=0.6)
ax1.scatter(df_opt['pit_lap'], df_opt['total_minutes'],
            c=colors_pit, s=100, edgecolor='black',
            linewidth=1.2, zorder=5)

opt_idx = df_opt['total_time'].idxmin()
ax1.scatter(df_opt.loc[opt_idx, 'pit_lap'],
            df_opt.loc[opt_idx, 'total_minutes'],
            s=400, marker='*', color='gold',
            edgecolor='black', linewidth=2,
            zorder=10, label='Optimal')

ax1.annotate(
    f"Optimal: Lap {int(df_opt.loc[opt_idx,'pit_lap'])}\n"
    f"{df_opt.loc[opt_idx,'total_minutes']:.2f} min",
    xy=(df_opt.loc[opt_idx,'pit_lap'],
        df_opt.loc[opt_idx,'total_minutes']),
    xytext=(df_opt.loc[opt_idx,'pit_lap'] + 4,
            df_opt.loc[opt_idx,'total_minutes'] + 0.3),
    arrowprops=dict(arrowstyle='->', color='black', lw=2),
    fontsize=10, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.4',
              facecolor='yellow', alpha=0.8)
)

ax1.set_xlabel('Tour du Pit Stop', fontweight='bold', fontsize=12)
ax1.set_ylabel('Temps Total (min)', fontweight='bold', fontsize=12)
ax1.set_title('Pit Window Optimization\nMEDIUM → HARD — VER — Bahrain 2025',
              fontweight='bold', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Comparaison stratégies
ax2 = axes[1]
colors_strat = ['gold' if i == 0 else 'steelblue'
                for i in range(len(results_strat))]

ax2.barh(results_strat['Strategy'],
         results_strat['Total Time (min)'],
         color=colors_strat, edgecolor='black', linewidth=1.2)
ax2.set_xlabel('Temps Total (min)', fontweight='bold', fontsize=12)
ax2.set_title('Comparaison des Stratégies\nVER — Bahrain 2025',
              fontweight='bold', fontsize=13)

for i, (_, row) in enumerate(results_strat.iterrows()):
    delta_str = f"+{row['Delta (s)']:.1f}s" if row['Delta (s)'] > 0 else "OPTIMAL"
    ax2.text(row['Total Time (min)'] + 0.05, i,
             delta_str, va='center', fontsize=9, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/strategy_analysis_bahrain2025.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique sauvegardé")

In [ ]:
# ═══════════════════════════════════════════════════
# CELLULE 6 — Analyse par pilote (VER vs NOR vs LEC)
# ═══════════════════════════════════════════════════

drivers_to_compare = ['VER', 'NOR', 'LEC']
colors_drivers     = ['#1E41FF', '#FF8700', '#DC0000']

# Stratégie optimale pour chaque pilote
optimal_strategy = [
    {'compound': 'MEDIUM', 'laps': 28},
    {'compound': 'HARD',   'laps': 29}
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Lap times au fil de la course
ax1 = axes[0]
for driver, color in zip(drivers_to_compare, colors_drivers):
    lap_times   = []
    current_lap = 1

    for stint in optimal_strategy:
        for tire_age in range(1, stint['laps'] + 1):
            lt = predict_lap_time(
                driver     = driver,
                compound   = stint['compound'],
                tire_age   = tire_age,
                lap_number = current_lap,
                race_name  = 'Bahrain Grand Prix',
                year       = 2025,
                position   = 3,
                total_laps = 57
            )
            lap_times.append(lt)
            current_lap += 1

    ax1.plot(range(1, len(lap_times) + 1), lap_times,
             label=driver, color=color, linewidth=2.5, alpha=0.85)

# Marquer le pit stop
ax1.axvline(x=28, color='gray', linestyle='--',
            linewidth=2, alpha=0.7, label='Pit stop (lap 28)')
ax1.set_xlabel('Tour', fontweight='bold', fontsize=12)
ax1.set_ylabel('Lap Time (s)', fontweight='bold', fontsize=12)
ax1.set_title('Évolution des Lap Times\nMEDIUM→HARD — Bahrain 2025',
              fontweight='bold', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: Temps total par pilote
ax2 = axes[1]
total_times = []
for driver in drivers_to_compare:
    res = sim.simulate_strategy(driver, optimal_strategy, position=3)
    total_times.append(res['total_min'])

bars = ax2.bar(drivers_to_compare, total_times,
               color=colors_drivers, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Temps Total (min)', fontweight='bold', fontsize=12)
ax2.set_title('Temps Total par Pilote\n(même stratégie MEDIUM→HARD)',
              fontweight='bold', fontsize=13)

min_time = min(total_times)
for bar, t, driver in zip(bars, total_times, drivers_to_compare):
    delta = t - min_time
    label = f"{t:.2f}min" if delta == 0 else f"{t:.2f}min\n(+{delta:.1f}s)"
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.05,
             label, ha='center', fontsize=10, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/driver_comparison_bahrain2025.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Graphique sauvegardé")

In [1]:
import sys
print(sys.executable)

c:\Users\DELL\f1-strategy-optimizer\venv\Scripts\python.exe
